# Ликбез: Компьютерное зрение с `🤗 Transformers`

Эта тетрадка — это подробное руководство по использованию библиотеки `transformers` от Hugging Face для задач компьютерного зрения. Мы пройдем три основных сценария:

1.  **Извлечение Признаков (Feature Extraction)**: Как использовать мощь предобученных моделей для получения векторных представлений (эмбеддингов) изображений и обучать на них простые ML-классификаторы.
2.  **Поиск по Схожести**: Как на основе эмбеддингов построить простой индекс для поиска похожих изображений.
3.  **Полноценное Дообучение (Fine-Tuning)**: Как дообучить всю модель целиком на новую задачу с помощью удобного API `Trainer`.

**Цель**: Показать, что `transformers` — это не только про текст, но и про картинки, и что это невероятно удобно.

## Подготовка окружения

Установим все необходимые библиотеки.

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn torch torchvision

## Часть 1: Извлечение признаков (Feature Extraction) с помощью CLIP

**Идея**: Вместо того чтобы обучать сложную нейросеть с нуля, мы возьмем очень мощную предобученную модель **CLIP**, прогоним через нее наши картинки кошек и собак и получим для каждой картинки вектор (эмбеддинг). А уже на этих векторах мы обучим простую логистическую регрессию, что гораздо быстрее и требует меньше данных.

### 1.1. Загрузка данных

In [ ]:
import torch
from PIL import Image
import os
import random
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from torchvision import transforms

# Загрузим и распакуем датасет
!wget -q https://storage.yandexcloud.net/yandex-research/courses/dogs_vs_cats_1000.zip -O dogs_vs_cats_1000.zip
!unzip -qn dogs_vs_cats_1000.zip
!ls dogs_vs_cats_1000 | wc -l  # должно быть 2000

Посмотрим на случайные примеры

In [ ]:
fig, axs = plt.subplots(1, 8, figsize=(16, 2))

fnames = [fn for fn in os.listdir('dogs_vs_cats_1000')]
for ax, fname in zip(axs.ravel(), random.choices(fnames, k=8)):
    img_ = Image.open(os.path.join('dogs_vs_cats_1000', fname))
    ax.imshow(img_)
    ax.set_title(f"{fname.split('.')[0]}")
    ax.tick_params(left = False,labelleft = False , labelbottom = False, bottom = False)
plt.tight_layout()

### 1.2. Загрузка модели для извлечения признаков

Здесь мы используем `CLIPVisionModel` — это именно **зрительная часть** знаменитой модели CLIP от OpenAI. Она идеально подходит для нашей задачи.

- **`CLIPImageProcessor`**: Это специальный препроцессор для CLIP. Он знает, как правильно изменить размер и нормализовать цвета для этой конкретной модели.
- **`CLIPVisionModel`**: Это загрузчик самой модели (только зрительного энкодера). Он вернет нам модель, которая на вход принимает картинку, а на выходе отдает ее векторное представление (эмбеддинг).

In [ ]:
from transformers import CLIPVisionModel, CLIPImageProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем устройство: {device}")

model_name = "openai/clip-vit-base-patch32"

# Загружаем препроцессор для CLIP
processor = CLIPImageProcessor.from_pretrained(model_name)

# Загружаем зрительную часть модели CLIP
embedding_model = CLIPVisionModel.from_pretrained(model_name).to(device)

### 1.3. Получение эмбеддингов

Теперь напишем цикл, который будет проходить по всем картинкам, обрабатывать их пачками (батчами) и извлекать эмбеддинги. 

In [ ]:
X_ = []  # Хранилище для эмбеддингов батчей
Y_ = []  # Хранилище для меток батчей

filenames = [fname for fname in os.listdir('dogs_vs_cats_1000')]
batch_size = 64
img_batch_list = []  # для накопления картинок в батч

embedding_model.eval() # Переключаем модель в режим оценки
with torch.no_grad(): # Отключаем расчет градиентов для ускорения
    for i, fname in enumerate(tqdm(filenames)):
        img = Image.open(os.path.join("dogs_vs_cats_1000", fname)).convert('RGB')
        img_batch_list.append(img)
        Y_.append(1 if fname.startswith("cat") else 0)

        if len(img_batch_list) == batch_size or i >= len(filenames) - 1:
            # 1. Препроцессинг: готовим батч картинок для модели
            inputs = processor(images=img_batch_list, return_tensors="pt").to(device)
            
            # 2. Прямой проход через модель
            outputs = embedding_model(**inputs)

            # 3. Извлечение эмбеддингов. `pooler_output` - это как раз векторное представление
            # для каждой картинки в батче. Переносим результат на CPU.
            embeddings = outputs.pooler_output.cpu()
            
            # Проверяем, что все соответствует ожиданиям
            assert isinstance(embeddings,  torch.Tensor) and embeddings.device.type == "cpu"
            assert embeddings.ndim == 2 and embeddings.shape[1] == 512
            X_.append(embeddings)
            img_batch_list = []  

# Объединяем все эмбеддинги в одну матрицу
X = np.concatenate(X_, axis = 0)
# Превращаем метки в numpy массив
Y = np.array(Y_[:len(X)])

print(X.shape, Y.shape, np.mean(Y))

assert X.ndim == 2 and X.shape[1] == 512
assert X.shape[0] == len(filenames)
assert Y.ndim == 1 and Y.shape[0] == X.shape[0]
assert 0.49 <= np.mean(Y) <= 0.51

### 1.4. Обучение простого классификатора

У нас есть матрица `X` (2000 картинок, каждая представлена вектором из 512 чисел) и вектор `Y` (метки 0 или 1). Это классическая задача машинного обучения! Обучим на ней логистическую регрессию, KNN и другие модели машинного обучения на ваш выбор

In [ ]:
# ВАШ КОД ЗДЕСЬ

## Часть 2: Построение индекса похожих произведений искусства

**Идея**: Раз уж мы умеем превращать картинки в векторы с помощью CLIP, мы можем измерить "расстояние" между этими векторами. Картинки с близкими векторами будут семантически похожи. Мы используем косинусное сходство — популярную метрику для векторов.

### 2.1. Загрузка данных (WikiArt)
Используем библиотеку `datasets` для удобной загрузки данных с Hugging Face Hub.

In [ ]:
from datasets import load_dataset

# Загружаем датасет, перемешиваем и берем сэмпл из 1000 изображений
dataset = load_dataset("lyubachuba/wikiart_5k", split="train")
dataset_sample = dataset.shuffle(seed=42).select(range(1000))

### 2.2. Извлечение эмбеддингов для произведений искусства
Процесс абсолютно идентичен тому, что мы делали для кошек и собак. Мы просто переиспользуем нашу модель `embedding_model` (CLIP) и `processor`.

In [ ]:
art_embeddings = []

embedding_model.eval()
with torch.no_grad():
    # Для ускорения будем обрабатывать батчами
    for i in tqdm(range(0, len(dataset_sample), batch_size)):
        batch = dataset_sample[i:i+batch_size]
        images = [img.convert("RGB") for img in batch['image']]

        inputs = processor(images=images, return_tensors="pt").to(device)
        outputs = embedding_model(**inputs)
        embeddings = outputs.pooler_output.cpu().numpy()
        art_embeddings.append(embeddings)

art_embeddings_matrix = np.concatenate(art_embeddings, axis=0)
print("Матрица эмбеддингов для WikiArt:", art_embeddings_matrix.shape)

### 2.3. Поиск похожих изображений
Напишем функцию, которая для выбранной картинки находит 5 самых похожих.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_images(image_query_url, embeddings_matrix, top_k=5):
    """Находит top_k самых похожих изображений для изображения по ссылке."""
    
    ## ВАШ КОД ЗДЕСЬ
    
    return most_similar_indices # возвращает индексы похожих картинок

def plot_similar_images(image_query_url, similar_indices, dataset_sample):
    """Отрисовывает картинку-запрос и найденные похожие картинки."""
    num_images = len(similar_indices) + 1
    plt.figure(figsize=(16, 4))
    
    # Картинка-запрос
    plt.subplot(1, num_images, 1)
    plt.imshow( ........) # ВАШ КОД ЗДЕСЬ
    plt.title("Запрос")
    plt.axis('off')
    
    # Похожие картинки
    for i, idx in enumerate(similar_indices):
        plt.subplot(1, num_images, i + 2)
        plt.imshow( ........) # ВАШ КОД ЗДЕСЬ
        plt.title(f"Топ-{i+1}")
        plt.axis('off')
    
    plt.show()

# Выберем случайную картинку для поиска
image_query_url = ........ # ВАШ КОД ЗДЕСЬ
similar_indices = find_similar_images(image_query_url, art_embeddings_matrix)
plot_similar_images(image_query_url, similar_indices, dataset_sample)

## Часть 3: Полноценное дообучение (Fine-Tuning) с помощью `transformers`

**Этот раздел остался без изменений, так как он демонстрирует другой, более сложный подход и использует другую модель, что делает ликбез более полным.**

Это самый мощный и гибкий подход. Мы не просто используем признаки, а **дообучаем саму модель**, меняя её веса, чтобы она лучше решала нашу конкретную задачу.

**Задача**: Возьмем забавную задачу — определение возрастной группы по фотографии человека.
**Инструменты**: `Trainer` API из библиотеки `transformers`. Это высокоуровневая обертка, которая прячет от нас сложный цикл обучения (training loop) и позволяет сфокусироваться на данных и модели.

### Ликбез по компонентам `transformers` для обучения

Процесс дообучения с `transformers` состоит из нескольких ключевых шагов, для каждого из которых есть свой класс:

1.  **Загрузка данных (`datasets`)**: Мы снова используем `datasets` для загрузки датасета с Hugging Face Hub.
2.  **Препроцессор (`AutoImageProcessor`)**: Как и раньше, он приводит картинки к нужному формату. Но теперь он будет применяться "на лету" во время обучения.
3.  **Модель (`AutoModelForImageClassification`)**: **Важно!** Теперь мы используем класс `...ForImageClassification`. Он автоматически добавляет к предобученной модели (например, ViT) новую "голову" — классификационный слой, который мы и будем обучать. 
4.  **Аргументы обучения (`TrainingArguments`)**: Это огромный класс-конфигуратор, где мы задаем все параметры обучения: скорость обучения, количество эпох, размер батча, где сохранять модель, как часто валидировать и т.д. 
5.  **Метрики (`evaluate`)**: Функция, которая будет считать качество нашей модели (например, accuracy) во время обучения.
6.  **Тренер (`Trainer`)**: Главный дирижер. Он берет модель, аргументы, данные, препроцессор, метрики и сам организует весь процесс обучения, валидации, сохранения и даже выгрузки модели на Hub.


### 3.1. Загрузка и подготовка данных

Мы будем использовать датасет `criteo/AgeGuess`. Он содержит изображения лиц и метки возрастных групп.

In [ ]:
from datasets import load_dataset

# Чтобы не качать весь датасет, ограничимся небольшим количеством примеров для демонстрации
train_ds = load_dataset("criteo/AgeGuess", split="train[:1000]")
test_ds = load_dataset("criteo/AgeGuess", split="validation[:200]")

# Посмотрим на структуру
print(train_ds)
print(train_ds[0])

Как мы видим, у нас есть картинка (`image`) и метка класса (`label`). Нам нужно создать словари для удобного сопоставления индекса класса и его названия.

In [ ]:
labels = train_ds.features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label

print("id2label:", id2label)

### 3.2. Препроцессинг

Мы будем использовать быструю и современную модель `google/vit-base-patch16-224-in21k`. Это Vision Transformer, который отлично подходит для дообучения.

Создадим функцию, которая будет применять к каждому изображению `ImageProcessor`. Эта функция также добавит к данным аугментацию (случайные изменения), чтобы модель лучше обобщалась.

In [ ]:
from transformers import AutoImageProcessor
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor

model_name_finetune = "google/vit-base-patch16-224-in21k"
processor_finetune = AutoImageProcessor.from_pretrained(model_name_finetune)

# Получаем параметры нормализации из препроцессора
normalize = Normalize(mean=processor_finetune.image_mean, std=processor_finetune.image_std)
size = (
    processor_finetune.size["shortest_edge"]
    if "shortest_edge" in processor_finetune.size
    else (processor_finetune.size["height"], processor_finetune.size["width"])
)

# Создаем аугментации для обучающей выборки
_transforms = Compose([RandomResizedCrop(size), ToTensor(), normalize])

def train_transforms(examples):
    examples["pixel_values"] = [_transforms(img.convert("RGB")) for img in examples["image"]]
    return examples

def val_transforms(examples):
    # Для валидации/теста просто меняем размер, без случайных кропов
    val_transform = Compose([transforms.Resize(size), ToTensor(), normalize])
    examples["pixel_values"] = [val_transform(img.convert("RGB")) for img in examples["image"]]
    return examples

# Применяем трансформации к датасетам
train_ds.set_transform(train_transforms)
test_ds.set_transform(val_transforms)

### 3.3. Загрузка модели

Теперь используем `AutoModelForImageClassification`. Мы передаем ему:
- `name`: Имя предобученной модели.
- `num_labels`: Количество классов в нашей задаче. Модель заменит свою старую "голову" на новую с нужным нам количеством выходов.
- `id2label` и `label2id`: Чтобы модель "знала", какой выход нейросети какому классу соответствует.
- `ignore_mismatched_sizes=True`: Этот флаг **обязателен**, чтобы разрешить замену "головы".

In [ ]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    model_name_finetune,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True, # Важно!
)

### 3.4. Настройка обучения (`TrainingArguments`)

In [ ]:
from transformers import TrainingArguments

model_name_short = model_name_finetune.split("/")[-1]
batch_size = 16

args = TrainingArguments(
    f"{model_name_short}-finetuned-age-guess",   # Название папки для сохранения модели
    remove_unused_columns=False,                 # Не удалять колонку 'image', она нам нужна для set_transform
    evaluation_strategy="epoch",                 # Оценивать качество в конце каждой эпохи
    save_strategy="epoch",                       # Сохранять модель в конце каждой эпохи
    learning_rate=5e-5,                           # Скорость обучения
    per_device_train_batch_size=batch_size,       # Размер батча для обучения
    per_device_eval_batch_size=batch_size,        # Размер батча для валидации
    num_train_epochs=3,                           # Количество эпох обучения
    warmup_ratio=0.1,                             # "Прогрев" скорости обучения
    logging_steps=10,                             # Как часто выводить лог с потерями
    load_best_model_at_end=True,                  # В конце загрузить лучшую версию модели
    metric_for_best_model="accuracy",           # Метрика для определения "лучшей" модели
)

### 3.5. Метрики и `Trainer`

Осталось определить функцию для подсчета метрик и собрать все вместе в `Trainer`.

In [ ]:
import evaluate
from transformers import Trainer, DefaultDataCollator

# Загружаем метрику accuracy
metric = evaluate.load("accuracy")

# Функция для вычисления метрик
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

# Data Collator - это helper-класс, который собирает отдельные эелементы датасета в батч. 
# DefaultDataCollator - стандартный вариант, который подходит в большинстве случаев.
data_collator = DefaultDataCollator()

# Создаем Trainer!
trainer = Trainer(
    model,                          # наша модель
    args,                           # аргументы обучения
    train_dataset=train_ds,         # обучающий датасет
    eval_dataset=test_ds,           # валидационный датасет
    tokenizer=processor_finetune,   # тут передаем процессор, чтобы он правильно готовил данные 
    compute_metrics=compute_metrics,# функция для метрик
    data_collator=data_collator,    # наш data collator
)

### 3.6. Запуск обучения

Все готово! Одна строчка кода запускает весь сложный процесс.

In [ ]:
trainer.train()

### 3.7. Оценка и использование модели

После обучения мы можем оценить финальное качество на тестовом наборе и, что самое главное, легко использовать модель для предсказаний.

In [ ]:
trainer.evaluate()

In [ ]:
import requests
from transformers import pipeline

# Загрузим картинку для примера
url = "https://huggingface.co/datasets/criteo/AgeGuess/resolve/main/data/0_100004_1993.jpg"
image = Image.open(requests.get(url, stream=True).raw)

display(image)

# Самый простой способ использовать модель - через pipeline.
# Мы можем передать напрямую модель, которая уже в памяти трейнера.
pipe = pipeline("image-classification", model=trainer.model, image_processor=processor_finetune, device=device)

outputs = pipe(image)
print(outputs)

**Вывод по Части 3**: Мы увидели, насколько мощным и удобным является `Trainer` API. Мы определили компоненты (данные, модель, аргументы), а библиотека сама позаботилась о цикле обучения, оптимизаторе, логировании, сохранении и оценке. Этот подход является стандартом де-факто для дообучения моделей из экосистемы Hugging Face.

## Итоговое заключение

Мы рассмотрели три разных способа применения предобученных моделей для компьютерного зрения с помощью `transformers`:

1.  **Feature Extraction (с CLIP)**: Быстро, просто, не требует много ресурсов. Идеально для старта, поиска и когда данных мало.
2.  **Similarity Search**: Естественное расширение первого подхода, очень полезное для рекомендательных систем и поиска.
3.  **Fine-Tuning (с ViT)**: Наиболее мощный метод, который позволяет адаптировать всю модель под вашу задачу и достичь максимального качества. `Trainer` API делает этот процесс удивительно простым.

Теперь вы готовы применять эти техники для решения ваших собственных задач в области компьютерного зрения!